####Ex 1→ StructType and StructField - define schemas explicitly
Defining schemas explicitly is the first rule of production pipelines. Never let Spark infer the schema from data — it's slow, costs an extra full scan, and often guesses types wrong. Know StructType inside out.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import *
from decimal import *

# Full schema — one StructField per column
# StructField(name, dataType, nullable)
schema = StructType([
    StructField("order_id",    IntegerType(),       False),  # NOT NULL
    StructField("cust_id",     IntegerType(),       False),  # NOT NULL
    StructField("product",     StringType(),        True),
    StructField("category",    StringType(),        True),
    StructField("price",       DoubleType(),        True),
    StructField("qty",         IntegerType(),       True),
    StructField("order_date",  DateType(),          True),
    StructField("is_returned", BooleanType(),       True),
    StructField("tax_rate",    DecimalType(5, 2),   True),   # precision=5, scale=2
    StructField("created_at",  TimestampType(),     True),
])

data = [
    (1001, 1, "Laptop",   "Electronics", 75000.0, 1, date(2024,1,10), False, Decimal(0.18), None),
    (1002, 2, "T-Shirt",  "Clothing",      999.0, 3, date(2024,1,15), False, Decimal(0.12), None),
    (1003, 1, "Rice 5kg", "Grocery",       450.0, 5, date(2024,2,1),  True,  Decimal(0.05), None),
    (1004, 3, "Phone",    "Electronics", 25000.0, 2, date(2024,2,10), False, Decimal(0.18), None),
]

df = spark.createDataFrame(data, schema)
df.display()
df.printSchema()

# Inspect schema programmatically
print("Column names:", df.columns)
print("dtypes:", df.dtypes)
print("Schema as JSON:", df.schema.json())

# Check a specific field
field = df.schema["price"]
print(f"price → type: {field.dataType}, nullable: {field.nullable}")

order_id,cust_id,product,category,price,qty,order_date,is_returned,tax_rate,created_at
1001,1,Laptop,Electronics,75000.0,1,2024-01-10,false,0.18,null
1002,2,T-Shirt,Clothing,999.0,3,2024-01-15,false,0.12,null
1003,1,Rice 5kg,Grocery,450.0,5,2024-02-01,true,0.05,null
1004,3,Phone,Electronics,25000.0,2,2024-02-10,false,0.18,null


root
 |-- order_id: integer (nullable = false)
 |-- cust_id: integer (nullable = false)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- qty: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- is_returned: boolean (nullable = true)
 |-- tax_rate: decimal(5,2) (nullable = true)
 |-- created_at: timestamp (nullable = true)

Column names: ['order_id', 'cust_id', 'product', 'category', 'price', 'qty', 'order_date', 'is_returned', 'tax_rate', 'created_at']
dtypes: [('order_id', 'int'), ('cust_id', 'int'), ('product', 'string'), ('category', 'string'), ('price', 'double'), ('qty', 'int'), ('order_date', 'date'), ('is_returned', 'boolean'), ('tax_rate', 'decimal(5,2)'), ('created_at', 'timestamp')]
Schema as JSON: {"fields":[{"metadata":{},"name":"order_id","nullable":false,"type":"integer"},{"metadata":{},"name":"cust_id","nullable":false,"type":"integer"},{"metadata":{},"name":"product","nullable":tr

####Ex 2→ inferSchema vs explicit schema - why explicit always wins
inferSchema scans the entire file before processing — doubling read time on large files. It also frequently guesses wrong types. This exercise proves both problems with a concrete example.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time

# Create a CSV in DBFS to demonstrate
csv_content = """
order_id,cust_id,product,price,qty,order_date,is_vip
1001,1,Laptop,75000.0,1,2024-01-10,true
1002,2,T-Shirt,999.0,3,2024-01-15,false
1003,1,Rice,450.0,5,2024-02-01,true
1004,3,Phone,25000.0,2,2024-02-10,false
"""

dbutils.fs.put("/Volumes/pyspark_course/default/datamaked/orders_demo_schema.csv", csv_content, overwrite=True)

# inferSchema - Spark scans the file twice
t0 = time.time()
df_infer = spark.read.format("csv") \
                     .option("header",True) \
                     .option("inferSchema",True) \
                     .load("/Volumes/pyspark_course/default/datamaked/orders_demo_schema.csv")
t1 = time.time()

print("inferSchema result:")
df_infer.printSchema()
print(f"Time: {(t1-t0)*1000:.0f}ms  (scanned file twice)")

# explicit schema - Spark reads the file once
schema = StructType([
    StructField("order_id",   IntegerType(), True),
    StructField("cust_id",    IntegerType(), True),
    StructField("product",    StringType(),  True),
    StructField("price",      DoubleType(),  True),
    StructField("qty",        IntegerType(), True),
    StructField("order_date", StringType(),  True),  # keep as string, cast later
    StructField("is_vip",     StringType(),  True),  # keep as string, cast later
])

t2 = time.time()
df_explicit = spark.read.format("csv") \
                        .option("header", "true") \
                        .schema(schema) \
                        .load("/Volumes/pyspark_course/default/datamaked/orders_demo_schema.csv")
t3 = time.time()

print("\n=== Explicit schema result ===")
df_explicit.printSchema()
print(f"Time: {(t3-t2)*1000:.0f}ms  (single scan)")

Wrote 210 bytes.
inferSchema result:
root
 |-- order_id: integer (nullable = true)
 |-- cust_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- price: double (nullable = true)
 |-- qty: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- is_vip: boolean (nullable = true)

Time: 0ms  (scanned file twice)

=== Explicit schema result ===
root
 |-- order_id: integer (nullable = true)
 |-- cust_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- price: double (nullable = true)
 |-- qty: integer (nullable = true)
 |-- order_date: string (nullable = true)
 |-- is_vip: string (nullable = true)

Time: 0ms  (single scan)


####Ex 3→ Type casting - convert column types safely
After reading raw data as strings (the safe approach), you need to cast columns to their correct types. Know how to cast safely — failed casts return null instead of crashing, which is exactly what you want in production.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Raw data — everything as string (common when reading messy CSVs)
raw = spark.createDataFrame([
    ("1001", "1", "75000.0",  "3", "2024-01-10", "true",  "18"),
    ("1002", "2",  "999.50",  "1", "15-01-2024", "false", "12"),  # different date format!
    ("1003", "3",     "abc",  "2", "2024-02-01", "yes",   "5"),   # bad price!
    ("1004", "4", "25000.0", "NA", "2024/02/10", "1",     "18"),  # bad qty! different date sep
    ("1005", "5", "45000.0",  "1",         None, "false", None),  # null date
], ["order_id","cust_id","price","qty","order_date","is_vip","tax_pct"])

raw.printSchema()  # all StringType
raw.display()

# ── Cast using .cast() ──
df_cast = raw.withColumn("order_id",  col("order_id").cast(IntegerType())) \
             .withColumn("cust_id",   col("cust_id") .cast(IntegerType())) \
             .withColumn("price",     col("price")   .cast(DoubleType())) \
             .withColumn("tax_pct",   col("tax_pct") .cast(DoubleType()))   # null stays null

# ── Cast booleans manually — "yes"/"1" aren't auto-cast ──
df_cast = df_cast.withColumn("is_vip",when(col("is_vip").isin("true","True","1","yes"), True) \
                 .when(col("is_vip").isin("false","False","0","no"), False) \
                 .otherwise(None)
)

# ── Cast dates — handle multiple formats ──
df_cast = df_cast.withColumn(
    "order_dt",
    when(
        col("order_date").rlike("\\d{4}-\\d{2}-\\d{2}"),
        to_date(col("order_date"), "yyyy-MM-dd")
    ) \
    .when(
        col("order_date").rlike("\\d{2}/\\d{2}/\\d{4}"),
        to_date(col("order_date"), "dd/MM/yyyy")
    ) \
    .when(
        col("order_date").rlike("\\d{4}/\\d{2}/\\d{2}"),
        to_date(col("order_date"), "yyyy/MM/dd")
    ) \
    .otherwise(None)
)

df_cast.select("order_id","price","is_vip","order_date","order_dt","tax_pct").display()
# Check for cast failures (nulls that weren't null before)
print("Rows where price cast failed (was not null, became null):")
df_cast.filter(col("price").isNull() & raw["price"].isNotNull()).display()

root
 |-- order_id: string (nullable = true)
 |-- cust_id: string (nullable = true)
 |-- price: string (nullable = true)
 |-- qty: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- is_vip: string (nullable = true)
 |-- tax_pct: string (nullable = true)



order_id,cust_id,price,qty,order_date,is_vip,tax_pct
1001,1,75000.0,3,2024-01-10,true,18
1002,2,999.50,1,15-01-2024,false,12
1003,3,abc,2,2024-02-01,yes,5
1004,4,25000.0,NA,2024/02/10,1,18
1005,5,45000.0,1,null,false,null


---------------------------------------------------------------------------
NumberFormatException                     Traceback (most recent call last)
File <command-8094501837248480>, line 46
     28 # ── Cast dates — handle multiple formats ──
     29 df_cast = df_cast.withColumn(
     30     "order_dt",
     31     when(
   (...)
     43     .otherwise(None)
     44 )
---> 46 df_cast.select("order_id","price","is_vip","order_date","order_dt","tax_pct").show()
     47 # Check for cast failures (nulls that weren't null before)
     48 print("Rows where price cast failed (was not null, became null):")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1156, in DataFrame.show(self, n, truncate, vertical)
   1155 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1156     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:

####Ex 4→ Null handling - detect,fill,drop, and audit
Nulls are the most common source of silent bugs in pipelines. This exercise covers the full null lifecycle — detecting where they are, filling them intelligently, dropping rows when required, and auditing null counts.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Dataset with intentional nulls
schema = StructType([
    StructField("emp_id",  IntegerType(), True),
    StructField("name",    StringType(),  True),
    StructField("dept",    StringType(),  True),
    StructField("salary",  DoubleType(),  True),
    StructField("bonus",   DoubleType(),  True),
    StructField("city",    StringType(),  True),
])
data = [
    (1, "Alice",   "Engineering", 95000.0, 5000.0, "Delhi"),
    (2, "Bob",     None,          72000.0, None,   "Mumbai"),
    (3, None,      "Engineering", 88000.0, 3000.0, "Bangalore"),
    (4, "Diana",   "HR",          None,    None,   None),
    (5, "Eve",     "Marketing",   78000.0, 2000.0, "Mumbai"),
    (6, "Frank",   "Engineering", 102000.0,None,   "Chennai"),
]
df = spark.createDataFrame(data, schema)
df.display()

# ── Step 1: Null audit — count nulls per column ──
print("=== NULL AUDIT ===")
null_audit = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
null_audit.display()

# Null percentage per column
total = df.count()
df.select([
    (sum(col(c).isNull().cast("int")) / total * 100)
    .alias(f"{c}_null_pct")
    for c in df.columns
]).display()

# ── Step 2: Fill nulls intelligently ──
df_filled = df.fillna({"dept": "Unknown", "city": "Unknown"}) \
             .fillna({"bonus": 0.0}) \
             .withColumn("salary",coalesce(col("salary"), lit(0.0))) \
             .withColumn("name",coalesce(col("name"), lit("Unknown")))

print("=== After filling nulls ===")
df_filled.display()

# ── Step 3: Drop rows where critical columns are null ──
df_clean = df.dropna(subset=["emp_id","name"])  # must have these
print(f"After dropna(subset): {df_clean.count()} rows (was {df.count()})")


# ── Step 4: Flag rows with any null (don't drop, mark for review) ──
df.withColumn("has_null",
    when(
        col("name").isNull() | col("dept").isNull() |
        col("salary").isNull() | col("city").isNull(), True
    ).otherwise(False)
).display()

emp_id,name,dept,salary,bonus,city
1,Alice,Engineering,95000.0,5000.0,Delhi
2,Bob,null,72000.0,null,Mumbai
3,null,Engineering,88000.0,3000.0,Bangalore
4,Diana,HR,null,null,null
5,Eve,Marketing,78000.0,2000.0,Mumbai
6,Frank,Engineering,102000.0,null,Chennai


=== NULL AUDIT ===


emp_id,name,dept,salary,bonus,city
0,1,1,1,3,1


emp_id_null_pct,name_null_pct,dept_null_pct,salary_null_pct,bonus_null_pct,city_null_pct
0.0,16.666666666666664,16.666666666666664,16.666666666666664,50.0,16.666666666666664


=== After filling nulls ===


emp_id,name,dept,salary,bonus,city
1,Alice,Engineering,95000.0,5000.0,Delhi
2,Bob,Unknown,72000.0,0.0,Mumbai
3,Unknown,Engineering,88000.0,3000.0,Bangalore
4,Diana,HR,0.0,0.0,Unknown
5,Eve,Marketing,78000.0,2000.0,Mumbai
6,Frank,Engineering,102000.0,0.0,Chennai


After dropna(subset): 5 rows (was 6)


emp_id,name,dept,salary,bonus,city,has_null
1,Alice,Engineering,95000.0,5000.0,Delhi,false
2,Bob,null,72000.0,null,Mumbai,true
3,null,Engineering,88000.0,3000.0,Bangalore,true
4,Diana,HR,null,null,null,true
5,Eve,Marketing,78000.0,2000.0,Mumbai,false
6,Frank,Engineering,102000.0,null,Chennai,false


####Ex 5→   ArrayType columns - arrays of values
Array columns hold multiple values per row — tags, product lists, event sequences, skill sets. Know how to read, query, and explode them. This comes up with JSON API data constantly.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Schema with array columns
arr_schema = StructType([
    StructField("emp_id",  IntegerType(),             True),
    StructField("name",    StringType(),               True),
    StructField("skills",  ArrayType(StringType()),    True),
    StructField("scores",  ArrayType(DoubleType()),    True),
])

data = [
    (1, "Alice",   ["Python","Spark","SQL"],          [92.0, 88.0, 95.0]),
    (2, "Bob",     ["SQL","Tableau"],                 [78.0, 85.0]),
    (3, "Charlie", ["Python","Spark","Kafka","Airflow"],[90.0,87.0,82.0,88.0]),
    (4, "Diana",   [],                                []),          # empty array
    (5, "Eve",     None,                              None),        # null array
]

df = spark.createDataFrame(data, arr_schema)
df.printSchema()
df.display()

# Query array columns
df.select("name",
    size("skills")                   .alias("num_skills"),
    array_contains("skills","Spark") .alias("knows_spark"),
    sort_array("skills")             .alias("skills_sorted"),
).display()

# explode — one row per array element (skips null/empty arrays)
print("=== explode ===")
df.select("name", explode("skills")  .alias("skill")).display()

# explode_outer — keeps null/empty arrays as null rows
print("=== explode_outer (keeps nulls) ===")
df.select("name", explode_outer("skills").alias("skill")).display()

# posexplode — includes the index position
print("=== posexplode (with position) ===")
df.select("name", posexplode("skills").alias("pos","skill")).display()

# aggregate back — collect rows into array
df.select("name", explode("skills").alias("skill")) \
  .groupBy("name") \
  .agg(collect_list("skill").alias("skills_back")) \
  .display()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- skills: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- scores: array (nullable = true)
 |    |-- element: double (containsNull = true)



emp_id,name,skills,scores
1,Alice,"List(Python, Spark, SQL)","List(92.0, 88.0, 95.0)"
2,Bob,"List(SQL, Tableau)","List(78.0, 85.0)"
3,Charlie,"List(Python, Spark, Kafka, Airflow)","List(90.0, 87.0, 82.0, 88.0)"
4,Diana,List(),List()
5,Eve,null,null


name,num_skills,knows_spark,skills_sorted
Alice,3,true,"List(Python, SQL, Spark)"
Bob,2,false,"List(SQL, Tableau)"
Charlie,4,true,"List(Airflow, Kafka, Python, Spark)"
Diana,0,false,List()
Eve,null,null,null


=== explode ===


name,skill
Alice,Python
Alice,Spark
Alice,SQL
Bob,SQL
Bob,Tableau
Charlie,Python
Charlie,Spark
Charlie,Kafka
Charlie,Airflow


=== explode_outer (keeps nulls) ===


name,skill
Alice,Python
Alice,Spark
Alice,SQL
Bob,SQL
Bob,Tableau
Charlie,Python
Charlie,Spark
Charlie,Kafka
Charlie,Airflow
Diana,null


=== posexplode (with position) ===


name,pos,skill
Alice,0,Python
Alice,1,Spark
Alice,2,SQL
Bob,0,SQL
Bob,1,Tableau
Charlie,0,Python
Charlie,1,Spark
Charlie,2,Kafka
Charlie,3,Airflow


name,skills_back
Alice,"List(Python, Spark, SQL)"
Bob,"List(SQL, Tableau)"
Charlie,"List(Python, Spark, Kafka, Airflow)"


####Ex 6→ from_json to to_json - parse JSON strings
JSON columns stored as strings are extremely common — Kafka messages, API responses, event logs. from_json parses a string column into a proper struct. to_json serialises a struct back to a string.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Simulate data where one column contains a JSON string
raw_events = spark.createDataFrame([
    (1, '{"product":"Laptop","price":75000,"qty":1,"category":"Electronics"}'),
    (2, '{"product":"T-Shirt","price":999,"qty":3,"category":"Clothing"}'),
    (3, '{"product":"Phone","price":25000,"qty":2,"category":"Electronics"}'),
    (4, '{"product":"Rice","price":450,"qty":5,"category":"Grocery"}'),
    (5, None),                                                               # null JSON
    (6, '{"product":"Watch","price":8999}'),                                 # missing fields
], ["event_id","event_payload"])

raw_events.display()

# ── Define schema for the JSON payload ──
payload_schema = StructType([
    StructField("product",  StringType(),  True),
    StructField("price",    DoubleType(),  True),
    StructField("qty",      IntegerType(), True),
    StructField("category", StringType(),  True),
])

# ── from_json — parse the JSON string into a struct ──
df_parsed = raw_events.withColumn("payload",
    from_json(col("event_payload"), payload_schema)
)

df_parsed.printSchema()
df_parsed.display()

# Access struct fields
df_final = df_parsed.select("event_id",
    col("payload.product") .alias("product"),
    col("payload.price")   .alias("price"),
    col("payload.qty")     .alias("qty"),
    col("payload.category").alias("category"),
)   .withColumn("order_value",
      col("price") * col("qty"))

df_final.show()

# ── to_json — serialise struct back to JSON string ──
df_final .withColumn("json_output", to_json(struct("product","price","qty","order_value"))) \
         .select("event_id","json_output") \
         .display()

# ── schema_of_json — infer schema from a sample JSON string ──
sample = '{"product":"Laptop","price":75000,"qty":1,"tags":["new","sale"]}'
inferred = schema_of_json(sample)
print("Inferred schema:", inferred)

event_id,event_payload
1,"{""product"":""Laptop"",""price"":75000,""qty"":1,""category"":""Electronics""}"
2,"{""product"":""T-Shirt"",""price"":999,""qty"":3,""category"":""Clothing""}"
3,"{""product"":""Phone"",""price"":25000,""qty"":2,""category"":""Electronics""}"
4,"{""product"":""Rice"",""price"":450,""qty"":5,""category"":""Grocery""}"
5,null
6,"{""product"":""Watch"",""price"":8999}"


root
 |-- event_id: long (nullable = true)
 |-- event_payload: string (nullable = true)
 |-- payload: struct (nullable = true)
 |    |-- product: string (nullable = true)
 |    |-- price: double (nullable = true)
 |    |-- qty: integer (nullable = true)
 |    |-- category: string (nullable = true)



event_id,event_payload,payload
1,"{""product"":""Laptop"",""price"":75000,""qty"":1,""category"":""Electronics""}","List(Laptop, 75000.0, 1, Electronics)"
2,"{""product"":""T-Shirt"",""price"":999,""qty"":3,""category"":""Clothing""}","List(T-Shirt, 999.0, 3, Clothing)"
3,"{""product"":""Phone"",""price"":25000,""qty"":2,""category"":""Electronics""}","List(Phone, 25000.0, 2, Electronics)"
4,"{""product"":""Rice"",""price"":450,""qty"":5,""category"":""Grocery""}","List(Rice, 450.0, 5, Grocery)"
5,null,null
6,"{""product"":""Watch"",""price"":8999}","List(Watch, 8999.0, null, null)"


+--------+-------+-------+----+-----------+-----------+
|event_id|product|  price| qty|   category|order_value|
+--------+-------+-------+----+-----------+-----------+
|       1| Laptop|75000.0|   1|Electronics|    75000.0|
|       2|T-Shirt|  999.0|   3|   Clothing|     2997.0|
|       3|  Phone|25000.0|   2|Electronics|    50000.0|
|       4|   Rice|  450.0|   5|    Grocery|     2250.0|
|       5|   NULL|   NULL|NULL|       NULL|       NULL|
|       6|  Watch| 8999.0|NULL|       NULL|       NULL|
+--------+-------+-------+----+-----------+-----------+



event_id,json_output
1,"{""product"":""Laptop"",""price"":75000.0,""qty"":1,""order_value"":75000.0}"
2,"{""product"":""T-Shirt"",""price"":999.0,""qty"":3,""order_value"":2997.0}"
3,"{""product"":""Phone"",""price"":25000.0,""qty"":2,""order_value"":50000.0}"
4,"{""product"":""Rice"",""price"":450.0,""qty"":5,""order_value"":2250.0}"
5,{}
6,"{""product"":""Watch"",""price"":8999.0}"


Inferred schema: Column<'schema_of_json({"product":"Laptop","price":75000,"qty":1,"tags":["new","sale"]})'>
